<a href="https://colab.research.google.com/github/nghff/vlm-pca-exercise/blob/main/Raphi_Task_Results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re # Added for regular expression matching

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import gc

import os
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Raphi_Task_Data/'

model_id = 'Qwen/Qwen3-VL-2B-Instruct'
#model_id = 'allenai/Molmo2-8B'
#odel_id = 'llava-hf/llava-1.5-7b-hf'

RESULTS_SAVE_DIR = os.path.join(DATA_DIR, model_id)
if not os.path.exists(RESULTS_SAVE_DIR):
    # construct directories
    os.makedirs(RESULTS_SAVE_DIR, exist_ok=True)


In [ ]:
gc.collect()

# Init Stuff

In [ ]:
layers_hooked = {
    'Qwen/Qwen3-VL-2B-Instruct': [
        'model.visual.blocks.1',                # early visual features extracted by visual encoder
        'model.visual.blocks.11',                # mid visual features extracted by visual encoder
        'model.visual.blocks.23',               # final visual information extracted from image by visual encoder

        'model.visual.merger',                  # visual features 'mapped' to global language space

        'model.visual.deepstack_merger_list.0', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)
        'model.visual.deepstack_merger_list.1', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)
        'model.visual.deepstack_merger_list.2', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)

        'model.language_model.layers.0',
        'model.language_model.layers.1',        # early language features extracted by decoder
        'model.language_model.layers.13',       # mid language features extracted by decoder
        'model.language_model.layers.20',
        'model.language_model.layers.27'       # final language features extracted by decoder
    ],
    'llava-hf/llava-1.5-7b-hf': [
        'model.vision_tower.vision_model.encoder.layers.1',
        'model.vision_tower.vision_model.encoder.layers.11',
        'model.vision_tower.vision_model.encoder.layers.23',

        'model.multi_modal_projector',

        'model.language_model.layers.0',
        'model.language_model.layers.1',
        'model.language_model.layers.15',
        'model.language_model.layers.23',
        'model.language_model.layers.31'
    ],
    'allenai/Molmo2-8B':[
        'model.vision_backbone.image_vit.transformer.resblocks.1',
        'model.vision_backbone.image_vit.transformer.resblocks.12',
        'model.vision_backbone.image_vit.transformer.resblocks.24',

        'model.vision_backbone.image_projector',

        'model.transformer.blocks.0',
        'model.transformer.blocks.1',
        'model.transformer.blocks.17',
        'model.transformer.blocks.26',
        'model.transformer.blocks.35'
    ]
}

attentions_grabbed = {
    'Qwen/Qwen3-VL-2B-Instruct': [
        0, 1, 13, 27
    ],
    'llava-hf/llava-1.5-7b-hf': [
        0, 1, 15, 30, 31
    ],
    'allenai/Molmo2-8B':[
        # eager attention not supported
    ]
}

for model_id_ in attentions_grabbed:
    layers_hooked[model_id_].extend([f'attention_{i}' for i in attentions_grabbed[model_id_]])
print(model_id)
print(layers_hooked[model_id])

In [ ]:
def save_csv(df, name):
    df_path = os.path.join(RESULTS_SAVE_DIR, f'{name}.csv')
    df.to_csv(df_path)

def load_df(name):
    df_path = os.path.join(RESULTS_SAVE_DIR, f'{name}.pkl')
    return pd.read_pickle(df_path)

In [ ]:
df = load_df('activation_data')
save_csv(df, 'activation_data')

df.head()

In [ ]:
eval_results_df = load_df('results')
save_csv(eval_results_df, 'results')

eval_results_df

In [ ]:
object_token_idx = None
for idx in range(1, len(df['input_tokens'][0])):
    if len(set([sequence[-idx] for sequence in df['input_tokens']])) != 1:
        object_token_idx = -idx
        break
print(object_token_idx)
print(df['input_tokens'][0][object_token_idx - 1: object_token_idx + 1])

#PCA

In [ ]:
def extract_vector(a, token_idx):
    if a.ndim == 1:
        return a
    if a.ndim >= 2:
        t = a.shape[0]
        if token_idx == "average":
            return np.mean(a, axis=0)
        else:
            idx = -1 if token_idx is None else int(token_idx)
            return a[idx]

    return a


def pca_layer_plots(
    df,
    layer_cols: list[str],
    target_col: str = 'target',
    token_idx: int | str | None = None,
    standardize: bool = True,
    n_components: int = 2,
    point_size: int = 18,
    title_prefix: str = "PCA",
):
    if target_col not in df.columns:
        raise ValueError(f'target column "{target_col}" is not found in the headers of given dataframe')

    counts = df[target_col].to_numpy()
    counts_groups = np.unique(counts)

    results = {}
    for col in layer_cols:
        if col not in df.columns:
            raise ValueError(f'layer column "{col}" is not found in the headers of given dataframe')

        vectors = []
        counts_l = []
        num_misfits = 0

        for v, count in zip(df[col].values, counts):
            vec = extract_vector(v, token_idx)

            vec = np.array(vec)
            if vec.ndim == 1:
                vec = vec.reshape(-1)
            if vec.size == 0 or np.any(np.isnan(vec)):
                num_misfits += 1
                print(f'misfit {col}: {v}: {vec}')
                continue
            counts_l.append(count)
            vectors.append(vec)

        if num_misfits > 0:
            print(f'Found {num_misfits} misfits in {col}')

        dims = [vec.shape[0] for vec in vectors]
        max_dim_size = max(dims)

        for i, vec in enumerate(vectors):
            if vec.shape[0] < max_dim_size:
                vectors[i] = np.pad(vec, (max_dim_size - vec.shape[0], 0), mode='constant')

        if standardize:
            vectors = StandardScaler().fit_transform(vectors)

        vector_mags = np.linalg.norm(vectors, axis=1)
        counts_l = np.array(counts_l)

        # filter outliers by magnitude
        vectors = vectors[vector_mags < np.percentile(vector_mags, 98)]
        counts_l = counts_l[vector_mags < np.percentile(vector_mags, 98)]

        pca = PCA(n_components=n_components, random_state=0)
        z_proj = pca.fit_transform(vectors)

        #plot
        fig, ax = plt.subplots(figsize=(10, 10))
        sc = ax.scatter(
            z_proj[:, 0],
            z_proj[:, 1],
            c=counts_l,
            cmap='viridis',
            s=point_size
        )
        ax.set_xlabel('PC 1')
        ax.set_ylabel('PC 2')
        ax.set_title(f'{title_prefix}: {col} (token_idx = {'last' if token_idx is None else token_idx})')
        cb = plt.colorbar(sc, ax=ax)
        cb.set_label(target_col)

        evr = pca.explained_variance_ratio_
        ax.text(
            0.05, 0.99,
            f'Explained Variance Ratio: {evr[0]:.2f}, {evr[1]:.2f}',
            transform=ax.transAxes,
            va='top',
            ha='left'
        )

        # save figure
        fig_path = os.path.join(RESULTS_SAVE_DIR, f'{col}_pca_token-{'last' if token_idx is None else token_idx}.png')
        plt.savefig(fig_path)
        plt.close()

        results[col] = {"pca": pca, "counts": counts_l, "X": vectors, "Z": z_proj, "fig": fig, "bad_rows": num_misfits}

    return results

In [ ]:
layers_hooked[model_id]


In [ ]:
results = pca_layer_plots(
    df,
    layer_cols=layers_hooked[model_id],
    target_col="target",
    token_idx=object_token_idx,
    standardize=True,
    n_components=2
)

In [ ]:
results = pca_layer_plots(
    df,
    layer_cols=layers_hooked[model_id],
    target_col="target",
    token_idx=None,
    standardize=True,
    n_components=2
)

In [ ]:
results = pca_layer_plots(
    df,
    layer_cols=layers_hooked[model_id],
    target_col="target",
    token_idx='average',
    standardize=True,
    n_components=2
)

# Plotting the Five Decoder Layers

In [ ]:
decoder_layer_ids = []
for layer_id in layers_hooked[model_id]:
    if layer_id.startswith('model.language_model.layers.') or layer_id.startswith('model.transformer.blocks.'):
        decoder_layer_ids.append(layer_id)
decoder_layer_ids

In [ ]:
print(f'old dims = {results[decoder_layer_ids[-1]]['X'].shape[-1]}')
print(f'new dims = {results[decoder_layer_ids[-1]]['Z'].shape[-1]}')

In [ ]:
# plot pca for decoder layers together
figsize = (10, 7)
fig, axes = plt.subplots(
    5,
    1,
    figsize=(figsize[0], figsize[1] * 5),
    constrained_layout=True
)

for i, layer_id in enumerate(decoder_layer_ids):
    ax = axes[i]
    ax.set_title(layer_id)
    sc = ax.scatter(
        results[layer_id]["Z"][:, 0],
        results[layer_id]["Z"][:, 1],
        c=results[layer_id]["counts"],
        cmap='viridis',
        s=18
    )
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    cb = plt.colorbar(sc, ax=ax)
    cb.set_label('target')

    evr = results[layer_id]["pca"].explained_variance_ratio_
    ax.text(
        0.05, 0.99,
        f'Explained Variance Ratio: {evr[0]:.2f}, {evr[1]:.2f}',
        transform=ax.transAxes,
        va='top',
        ha='left'
    )

# save
fig_path = os.path.join(RESULTS_SAVE_DIR, 'decoder_fivelayers_pca.png')
plt.savefig(fig_path)